# 01 - Ingesta

Pasamos los CSV de `/nyc/raw` a Parquet en `/nyc/bronze` sin cambiarles nada. Se corre una sola vez, después de `scripts/descargar_datos.sh`.

| Capa | Ruta | Contenido |
|---|---|---|
| cruda | `/nyc/raw/<fuente>/` | CSV originales |
| bronce | `/nyc/bronze/<fuente>/` | igual pero en Parquet, todo texto (este notebook) |
| plata | `/nyc/silver/<fuente>/` | tipos, filtros y limpieza (04 y 05) |
| oro | `/nyc/gold/<tabla>/` | agregados para el análisis y los modelos |

Usamos Parquet porque guarda los datos por columnas, viene comprimido y Spark lo puede leer en paralelo.

## 1. Sesión y rutas

In [1]:
import re
import time
import pandas as pd
from pyspark.sql import functions as F

from config import RAW, BRONZE, sesion

spark = sesion("01_ingesta")
sc = spark.sparkContext

# HDFS desde python
jvm = spark._jvm
fs = jvm.org.apache.hadoop.fs.FileSystem.get(sc._jsc.hadoopConfiguration())
Path = jvm.org.apache.hadoop.fs.Path

def tamano_hdfs(ruta):
    return fs.getContentSummary(Path(ruta)).getLength()

print("Master:", sc.master, "| paralelismo:", sc.defaultParallelism)
print("HDFS  :", fs.getUri())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/27 16:04:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Master: spark://10.43.97.45:7077 | paralelismo: 6
HDFS  : hdfs://10.43.97.45:9000


## 2. Manifiesto

El manifiesto lo deja el script de descarga: fecha, tamaño y número de líneas de cada archivo.

In [2]:
manifiesto = (
    spark.read.option("header", True).option("sep", "\t")
    .csv(f"{RAW}/manifiesto.tsv")
    .toPandas()
    .astype({"bytes": "int64", "lineas": "int64"})
)
manifiesto["tamano_MB"] = (manifiesto["bytes"] / 1024**2).round(1)
manifiesto[["fuente", "id", "descargado_utc", "tamano_MB", "lineas"]]

,fuente,id,descargado_utc,tamano_MB,lineas
0,arrestos_historico,8h9b-rp9u,2026-09-27T17:33:01Z,1279.3,6264979
1,arrestos_2026,uip8-fykc,2026-09-27T17:33:48Z,24.9,141871
2,choques,h9gi-nx95,2026-09-27T17:44:41Z,454.5,2269189
3,vehiculos,bm4k-52h4,2026-09-27T17:55:17Z,803.6,4551003
4,pobreza,cts7-vksw,2026-09-27T17:55:40Z,13.1,68274
5,sat_2012,f9bf-2cp4,2026-09-27T17:55:48Z,0.0,479
6,puma_2010,k2r4-n9ax,2026-09-27T17:55:56Z,3.5,4


## 3. Lectura

- Todo se lee como texto, sin `inferSchema`. Si no, se pierden cosas como los ceros a la izquierda de los códigos postales, y además los puntajes del SAT traen `"s"`. Los tipos se ponen después, en plata.
- Hay campos con saltos de línea, por eso va `multiLine=True`. El problema es que así cada CSV lo lee una sola tarea, entonces antes de escribir se hace un `repartition`.
- Los nombres de las columnas quedan en minúscula y con guion bajo.

In [3]:
def normalizar(nombre):
    n = nombre.strip().lower()
    n = re.sub(r"[^0-9a-z]+", "_", n).strip("_")
    return n or "columna"

def normalizar_columnas(df):
    vistos, nuevos = {}, []
    for c in df.columns:
        n = normalizar(c)
        if n in vistos:          # por si se repiten
            vistos[n] += 1
            n = f"{n}_{vistos[n]}"
        else:
            vistos[n] = 0
        nuevos.append(n)
    return df.toDF(*nuevos)

[normalizar(c) for c in ["CRASH DATE", "NUMBER OF PERSONS INJURED", "Lon_Lat", "SAT Math Avg. Score"]]

['crash_date', 'number_of_persons_injured', 'lon_lat', 'sat_math_avg_score']

## 4. CSV a Parquet

Más o menos una partición por cada 128 MB, y nunca menos particiones que cores.

In [4]:
FUENTES_CSV = [f for f in manifiesto["fuente"] if f != "puma_2010"]   # puma es geojson

def ingerir(fuente):
    t0 = time.time()
    df = (
        spark.read
        .option("header", True)
        .option("multiLine", True)
        .option("escape", '"')
        .option("inferSchema", False)
        .csv(f"{RAW}/{fuente}/")
    )
    df = normalizar_columnas(df)
    bytes_csv = tamano_hdfs(f"{RAW}/{fuente}")
    particiones = max(sc.defaultParallelism, int(bytes_csv / (128 * 1024**2)) + 1)
    df.repartition(particiones).write.mode("overwrite").parquet(f"{BRONZE}/{fuente}")

    bronce = spark.read.parquet(f"{BRONZE}/{fuente}")
    filas = bronce.count()
    lineas = int(manifiesto.loc[manifiesto["fuente"] == fuente, "lineas"].iloc[0])
    bytes_pq = tamano_hdfs(f"{BRONZE}/{fuente}")
    return {
        "fuente": fuente,
        "filas": filas,
        "columnas": len(bronce.columns),
        "lineas_csv": lineas,
        "registros_multilinea": (lineas - 1) - filas,
        "CSV_MB": round(bytes_csv / 1024**2, 1),
        "Parquet_MB": round(bytes_pq / 1024**2, 1),
        "compresion_x": round(bytes_csv / bytes_pq, 1),
        "particiones": particiones,
        "segundos": round(time.time() - t0, 1),
    }

resumen = []
for fuente in FUENTES_CSV:
    r = ingerir(fuente)
    print(f"{fuente:>20}: {r['filas']:>12,} filas en {r['segundos']} s")
    resumen.append(r)

  arrestos_historico:    6,264,978 filas en 93.8 s


       arrestos_2026:      141,870 filas en 4.5 s


26/09/27 16:06:13 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


             choques:    2,269,187 filas en 37.2 s


           vehiculos:    4,551,002 filas en 47.7 s


             pobreza:       68,273 filas en 4.6 s


            sat_2012:          478 filas en 2.0 s


In [5]:
resumen_df = pd.DataFrame(resumen)
resumen_df

,fuente,filas,columnas,lineas_csv,registros_multilinea,CSV_MB,Parquet_MB,compresion_x,particiones,segundos
0,arrestos_historico,6264978,19,6264979,0,1279.3,371.2,3.4,10,93.8
1,arrestos_2026,141870,19,141871,0,24.9,5.8,4.3,8,4.5
2,choques,2269187,29,2269189,1,454.5,87.9,5.2,8,37.2
3,vehiculos,4551002,25,4551003,0,803.6,211.9,3.8,8,47.7
4,pobreza,68273,61,68274,0,13.1,5.0,2.6,8,4.6
5,sat_2012,478,6,479,0,0.0,0.0,0.7,8,2.0


`registros_multilinea` es la diferencia entre las líneas del archivo y los registros que leyó Spark. Si da más de 0 es porque hay campos con saltos de línea.

## 5. Esquema bronce

In [6]:
for fuente in FUENTES_CSV:
    df = spark.read.parquet(f"{BRONZE}/{fuente}")
    print(f"\n=== {fuente} ({len(df.columns)} columnas) ===")
    print(", ".join(df.columns))


=== arrestos_historico (19 columnas) ===
arrest_key, arrest_date, pd_cd, pd_desc, ky_cd, ofns_desc, law_code, law_cat_cd, arrest_boro, arrest_precinct, jurisdiction_code, age_group, perp_sex, perp_race, x_coord_cd, y_coord_cd, latitude, longitude, lon_lat



=== arrestos_2026 (19 columnas) ===
arrest_key, arrest_date, pd_cd, pd_desc, ky_cd, ofns_desc, law_code, law_cat_cd, arrest_boro, arrest_precinct, jurisdiction_code, age_group, perp_sex, perp_race, x_coord_cd, y_coord_cd, latitude, longitude, location

=== choques (29 columnas) ===
crash_date, crash_time, borough, zip_code, latitude, longitude, location, on_street_name, cross_street_name, off_street_name, number_of_persons_injured, number_of_persons_killed, number_of_pedestrians_injured, number_of_pedestrians_killed, number_of_cyclist_injured, number_of_cyclist_killed, number_of_motorist_injured, number_of_motorist_killed, contributing_factor_vehicle_1, contributing_factor_vehicle_2, contributing_factor_vehicle_3, contributing_factor_vehicle_4, contributing_factor_vehicle_5, collision_id, vehicle_type_code_1, vehicle_type_code_2, vehicle_type_code_3, vehicle_type_code_4, vehicle_type_code_5



=== vehiculos (25 columnas) ===
unique_id, collision_id, crash_date, crash_time, vehicle_id, state_registration, vehicle_type, vehicle_make, vehicle_model, vehicle_year, travel_direction, vehicle_occupants, driver_sex, driver_license_status, driver_license_jurisdiction, pre_crash, point_of_impact, vehicle_damage, vehicle_damage_1, vehicle_damage_2, vehicle_damage_3, public_property_damage, public_property_damage_type, contributing_factor_1, contributing_factor_2



=== pobreza (61 columnas) ===
serialno, sporder, pwgtp, wgtp, agep, cit, rel, sch, schg, schl, sex, esr, lanx, eng, msp, mar, wkw, wkhp, dis, jwtr, np, ten, hht, agecateg, boro, citizenstatus, educattain, est_childcare, est_commuting, est_eitc, est_ficatax, est_heap, est_housing, est_incometax, est_moop, est_nutrition, est_povgap, est_povgapindex, ethnicity, famtype_pu, ftptwork, intp_adj, mrgp_adj, nycgov_income, nycgov_pov_stat, nycgov_rel, nycgov_threshold, off_pov_stat, off_threshold, oi_adj, pa_adj, povunit_id, povunit_rel, pretaxincome_pu, retp_adj, rntp_adj, semp_adj, ssip_adj, ssp_adj, totalworkhrs_pu, wagp_adj

=== sat_2012 (6 columnas) ===
dbn, school_name, num_of_sat_test_takers, sat_critical_reading_avg_score, sat_math_avg_score, sat_writing_avg_score


In [7]:
# ejemplo
spark.read.parquet(f"{BRONZE}/arrestos_historico").limit(5).toPandas()

,arrest_key,arrest_date,pd_cd,pd_desc,ky_cd,ofns_desc,law_code,law_cat_cd,arrest_boro,arrest_precinct,jurisdiction_code,age_group,perp_sex,perp_race,x_coord_cd,y_coord_cd,latitude,longitude,lon_lat
0,147056919,10/19/2015,792,WEAPONS POSSESSION 1 & 2,118,DANGEROUS WEAPONS,PL 265031B,F,Q,102,0,18-24,M,BLACK,1030920.0,189268.0,40.68605888600007,-73.83172338199995,POINT (-73.83172338199995 40.68605888600007)
1,89621631,03/14/2013,201,"TRESPASS 4,CRIMINAL",678,MISCELLANEOUS PENAL LAW,PL 1400500,V,M,14,3,25-44,M,BLACK,986882.0,214802.0,40.75626620700007,-73.99050124799999,POINT (-73.99050124799999 40.75626620700007)
2,86841429,09/20/2012,478,"THEFT OF SERVICES, UNCLASSIFIED",343,OTHER OFFENSES RELATED TO THEFT,PL 1651503,M,M,24,0,25-44,F,WHITE,994170.0,227958.0,40.792370809000026,-73.96417580099995,POINT (-73.96417580099995 40.792370809000026)
3,171951676,11/25/2017,339,"LARCENY,PETIT FROM OPEN AREAS,UNCLASSIFIED",341,PETIT LARCENY,PL 1552500,M,K,68,0,25-44,M,WHITE HISPANIC,976676.0,165942.0,40.62215389700003,-74.02728485599994,POINT (-74.02728485599994 40.62215389700003)
4,265465842,03/22/2023,101,ASSAULT 3,344,ASSAULT 3 & RELATED OFFENSES,PL 1200001,M,B,40,0,25-44,M,BLACK,1005040,234533,40.810391,-73.924895,POINT (-73.924895 40.810391)


In [8]:
resumen_df.to_csv("../figuras/tabla_ingesta.csv", index=False)

## Conclusiones

- Los CSV originales siguen intactos en `/nyc/raw`.
- Bronce tiene lo mismo pero en Parquet, que pesa menos y se lee más rápido.
- El filtro por periodo todavía no se aplica; eso se hace al pasar a plata.